# Experiment 1: priming / persona → KV change (from stats files)

Same analyses as `exp1_priming.ipynb`, read from the per-passage stats files written by `kvmeaning.stats` (`KV_STATS` = one design's stats dir) instead of raw tensors. Tensors are loaded lazily per passage and layer.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from kvmeaning.stats import StatsReader

STATS = Path(os.environ['KV_STATS'])
FIG = Path(os.environ.get('KV_FIG', f'../figures/exp1_stats/{STATS.name}'))
FIG.mkdir(parents=True, exist_ok=True)
S = StatsReader(STATS)
REPS = ('v', 'k_pre', 'k_post', 'resid')
KV = REPS[:3]
PIDS = S.pids
M0 = S.meta(PIDS[0])
EFFECT = tuple(c for c in ('H+', 'H-', 'C+', 'C-', 'P1', 'P2') if c in M0['conds'])
CONDS = ('F1', 'F2') + EFFECT
PAIRS = [('F1', 'F2')] + [('F1', c) for c in EFFECT] + [('P1', 'P2')]
LAB = {'F1': 0, 'F2': 0, 'P1': 1, 'P2': 2}
STEP, TB, NL, LAYERS_SHOW = M0['STEP'], M0['TB'], M0['NL'], M0['LAYERS_SHOW']
DIST, MD, RES, RSCALE, PF, LOGITS = S.DIST, S.MD, S.RES, S.RSCALE, S.PF, S.LOGITS
print(STATS, len(PIDS), 'passages', 'layers', NL, 'body tokens', TB)


In [ ]:
def mean_over_passages(r, a, b, k):
    return torch.stack([DIST[pid, r, a, b][k] for pid in PIDS]).mean(0)  # [L, (H,) T]


# Layer 0: body-token inputs are identical in every condition, so K/V there depend on the prefix only
# through nothing; resid index 0 is the embedding output.
for r in KV:
    print(r, 'layer-0 max abs diff F1 vs P1:', float(mean_over_passages(r, 'F1', 'P1', 0)[0].max()))

## 1. Layer×head heatmap (distance / noise floor)

In [ ]:
def ratio(eff, floor, eps=1e-6):
    return torch.where(floor > eps, eff / floor.clamp_min(eps), torch.full_like(eff, float('nan')))


fig, axes = plt.subplots(len(KV), len(EFFECT), figsize=(14, 12), sharey=True, gridspec_kw=dict(hspace=0.45))
for i, r in enumerate(KV):
    floor = mean_over_passages(r, 'F1', 'F2', 0).mean(-1)
    ratios = {c: ratio(mean_over_passages(r, 'F1', c, 0).mean(-1), floor) for c in EFFECT}
    lim = max(float(np.nanmax(np.abs(np.log10(v.numpy())))) for v in ratios.values())
    for j, c in enumerate(EFFECT):
        ax = axes[i, j]
        im = ax.imshow(np.log10(ratios[c].numpy()), origin='lower', aspect='auto', cmap='RdBu_r', vmin=-lim, vmax=lim)
        ax.set_title(f'{r}: F1 vs {c}')
        ax.set_xlabel('KV head')
        if j == 0:
            ax.set_ylabel('layer')
    fig.colorbar(im, ax=axes[i, :], label='log10(L2 / noise floor)')
fig.savefig(FIG / 'heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

## 2. Layer curve

In [ ]:
fig, axes = plt.subplots(2, len(REPS), figsize=(16, 7), sharex=True)
LAYER_CURVE = {}
for j, r in enumerate(REPS):
    for k, name in enumerate(('L2 / noise floor', 'cosine dist / noise floor')):
        def lc(a, b):
            x = mean_over_passages(r, a, b, k)
            return x.mean(dim=tuple(range(1, x.dim()))) if r == 'resid' else x.mean(dim=(1, 2))
        floor = lc('F1', 'F2')
        ax = axes[k, j]
        for c in EFFECT:
            y = ratio(lc('F1', c), floor)
            LAYER_CURVE[r, k, c] = y
            ax.plot(range(len(y)), y, label=c)
        ax.axhline(1, color='k', lw=0.7, ls='--')
        ax.set_yscale('log')
        ax.set_title(f'{r}')
        ax.set_ylabel(name)
        ax.set_xlabel('layer (resid: 0 = embeddings)')
axes[0, 0].legend()
fig.savefig(FIG / 'layer_curve.png', dpi=130, bbox_inches='tight')
plt.show()
for r in REPS:
    print(r, 'L2 ratio at last layer:', {c: round(float(LAYER_CURVE[r, 0, c][-1]), 2) for c in EFFECT},
          'max over layers:', {c: round(float(np.nanmax(LAYER_CURVE[r, 0, c].numpy())), 2) for c in EFFECT})

## 3. Within-body decay curve

In [ ]:
def smooth(y, w=9):
    return np.convolve(y, np.ones(w) / w, mode='valid')


fig, axes = plt.subplots(1, len(REPS), figsize=(16, 3.8), sharey=False)
for j, r in enumerate(REPS):
    def by_pos(a, b):
        x = mean_over_passages(r, a, b, 0)[1:]  # drop layer 0 (identical in all conditions)
        return x.mean(dim=tuple(range(x.dim() - 1))).numpy()
    floor = by_pos('F1', 'F2')
    for c in EFFECT:
        axes[j].plot(smooth(by_pos('F1', c) / floor), label=c)
    axes[j].axhline(1, color='k', lw=0.7, ls='--')
    axes[j].set_title(r)
    axes[j].set_xlabel('body token position (moving avg 9)')
axes[0].set_ylabel('L2 / noise floor (layers 1+)')
axes[0].legend()
fig.savefig(FIG / 'decay.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Direction vectors and cross-passage cosine

In [ ]:
DIRS = {(r, c): torch.stack([MD[pid, r, c] for pid in PIDS]) for r in ('v', 'resid') for c in (*EFFECT, 'null')}  # [P, L, F]
# Null direction: F1 - F2 has no semantic content, so its cross-passage cosine is the chance level.


def cross_cos(d):
    u = F.normalize(d, dim=-1)
    n = len(PIDS)
    g = torch.einsum('plf,qlf->pql', u, u)
    off = ~torch.eye(n, dtype=torch.bool)
    return g[off].mean(0)  # [L]


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, r in zip(axes, ('v', 'resid')):
    for c in (*EFFECT, 'null'):
        ax.plot(range(DIRS[r, c].shape[1]), cross_cos(DIRS[r, c]), label=c, ls=':' if c == 'null' else '-')
    ax.set_title(f'{r}: cross-passage cosine of mean(cond - F)', fontsize=10)
    ax.set_xlabel('layer')
axes[0].legend()
fig.savefig(FIG / 'direction_cosine.png', dpi=130, bbox_inches='tight')
plt.show()
for r in ('v', 'resid'):
    print(r, {c: round(float(cross_cos(DIRS[r, c])[1:].mean()), 2) for c in (*EFFECT, 'null')}, '(mean over layers 1+)')

In [ ]:
# Per-token projection on the leave-one-passage-out persona direction (resid space, in units of F1 mean norm).
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for j, (ax, l) in enumerate(zip(axes, LAYERS_SHOW)):
    for c in ('P1', 'P2', *[c for c in EFFECT if c not in ('P1', 'P2')], 'null'):
        curves = []
        for i, pid in enumerate(PIDS):
            others = [k for k in range(len(PIDS)) if k != i]
            u = F.normalize(DIRS['resid', 'P1'][others, l].mean(0), dim=-1)
            R = {k: v[j].float() for k, v in RES[pid].items()}
            ref = R['F2'] if c == 'null' else 0.5 * (R['F1'] + R['F2'])
            src = R['F1' if c == 'null' else c]
            curves.append(((src - ref) @ u) / RSCALE[pid][j])
        ax.plot(smooth(torch.stack(curves).mean(0).numpy()), label=c, ls=':' if c == 'null' else '-')
    ax.set_title(f'resid index {l} (0 = embeddings)')
    ax.set_xlabel('body token position')
axes[0].set_ylabel('projection on P1 direction')
axes[0].legend()
fig.savefig(FIG / 'direction_projection.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Linear probe per layer (passage-level split)

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

NFOLD = min(5, len(PIDS))


def probe_layer(r, l):
    X, y, g = [], [], []
    for pi, pid in enumerate(PIDS):
        for c, lab in LAB.items():
            f = PF[pid, c, r][l].float().numpy()
            X.append(f); y += [lab] * len(f); g += [pi] * len(f)
    X, y, g = np.concatenate(X), np.array(y), np.array(g)
    accs = []
    for tr, te in GroupKFold(NFOLD).split(X, y, g):
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=300, class_weight='balanced'))
        clf.fit(X[tr], y[tr])
        accs.append(balanced_accuracy_score(y[te], clf.predict(X[te])))
    return float(np.mean(accs)), float(np.std(accs))


PROBE = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    for r in ('v', 'k_pre', 'resid'):
        PROBE[r] = np.array([probe_layer(r, l) for l in range(NL + (r == 'resid'))])

fig, ax = plt.subplots(figsize=(7, 4))
for r, a in PROBE.items():
    ax.errorbar(range(len(a)), a[:, 0], yerr=a[:, 1], label=r, capsize=2)
ax.axhline(1 / 3, color='k', ls='--', lw=0.7)
ax.set_xlabel('layer')
ax.set_ylabel(f'balanced accuracy (F / P1 / P2), {NFOLD}-fold by passage')
ax.legend()
fig.savefig(FIG / 'probe.png', dpi=130, bbox_inches='tight')
plt.show()
for r, a in PROBE.items():
    print(r, 'layer0 acc', round(a[0, 0], 3), 'best', round(a[:, 0].max(), 3), 'at layer', int(a[:, 0].argmax()))

## 6. Behavioral link: next-token KL

In [ ]:
# logits.body_last is the next-token distribution at the last body token (logits_last, the assistant-turn
# start, has <think> at p > 0.999 in every condition and cannot show any effect).
BODY_END = LOGITS
print('assistant-turn start is saturated; using logits.body_last for', len(PIDS), 'passages')

In [ ]:
def kl(p_logits, q_logits):
    lp, lq = F.log_softmax(p_logits, -1), F.log_softmax(q_logits, -1)
    return float((lp.exp() * (lp - lq)).sum())


KL_PAIRS = [('F1', c) for c in ('F2', *EFFECT)] + [('P1', 'P2')]
KL = {(a, b): [kl(BODY_END[pid, b], BODY_END[pid, a]) for pid in PIDS] for a, b in KL_PAIRS}  # KL(b || a)
print('KL(cond || ref) in nats at the last body token')
for (a, b), v in KL.items():
    print(f'{b} vs {a:3s}  mean {np.mean(v):.3f}  sd {np.std(v):.3f}  median {np.median(v):.3f}  > F2 floor in {np.mean(np.array(v) > np.array(KL["F1", "F2"])):.0%} of passages')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f'{b}|{a}' for a, b in KL_PAIRS], [np.mean(v) for v in KL.values()],
       yerr=[np.std(v) for v in KL.values()], capsize=3)
ax.set_ylabel('KL (nats)')
fig.savefig(FIG / 'kl.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. KV patching

See `exp1_patching.ipynb` (body region) and `exp1_patching_prefix.ipynb` (prefix region).

## Results summary

See the run's report; the 0.6B numbers in `exp1_priming.ipynb` do not apply to this notebook.